In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os

# Set up CDS API credentials
cds_config = """url: https://cds.climate.copernicus.eu/api
key: YOUR-API-KEY-HERE
"""

with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(cds_config)

print("CDS credentials configured")

CDS credentials configured


In [2]:
!pip install cdsapi -q

import cdsapi
import xarray as xr
import numpy as np
import gc

c = cdsapi.Client()
print("All imports successful")
print("CDS client ready")

All imports successful
CDS client ready


In [4]:
def download_and_aggregate(year, client):
    """
    Downloads all 24 hourly ERA5 precipitation values
    for one monsoon season (June-September),
    sums to true daily totals in mm/day,
    saves as a small daily NetCDF file,
    deletes the large hourly file immediately.
    """
    hours    = [f'{h:02d}:00' for h in range(24)]
    raw_file = f'/kaggle/working/era5_hourly_{year}.nc'
    day_file = f'/kaggle/working/era5_daily_{year}.nc'

    # Download hourly data
    print(f"\nDownloading {year} hourly data...")
    client.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type' : 'reanalysis',
            'variable'     : 'total_precipitation',
            'year'         : str(year),
            'month'        : ['06', '07', '08', '09'],
            'day'          : [str(d).zfill(2) for d in range(1, 32)],
            'time'         : hours,
            'format'       : 'netcdf',
            'area'         : [40, 60, 5, 100],  # N, W, S, E — India
            'grid'         : [1.0, 1.0],
        },
        raw_file
    )

    # Aggregate 24 hourly values → 1 daily total
    print(f"  Aggregating hourly → daily...")
    ds_hr  = xr.open_dataset(raw_file)
    ds_day = ds_hr['tp'].resample(valid_time='1D').sum() * 1000  # m → mm

    print(f"  Hourly shape : {ds_hr['tp'].shape}")
    print(f"  Daily shape  : {ds_day.shape}")
    print(f"  Daily max    : {float(ds_day.max()):.2f} mm/day")
    print(f"  Daily mean   : {float(ds_day.mean()):.2f} mm/day")

    # Save daily file
    ds_day.to_netcdf(day_file)
    print(f"  Saved → {day_file}")

    # Delete hourly file immediately to save space
    ds_hr.close()
    os.remove(raw_file)
    del ds_hr
    del ds_day
    gc.collect()

    print(f"  Hourly file deleted")
    print(f"  Year {year} complete ✅")

    return day_file

In [5]:
years_all  = list(range(1998, 2021))

already_done = []
to_download  = []

for year in years_all:
    f = f'/kaggle/working/era5_daily_{year}.nc'
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024*1024)
        already_done.append(year)
        print(f"  ✅ {year} already done ({size_mb:.1f} MB)")
    else:
        to_download.append(year)
        print(f"  ⏳ {year} needed")

print(f"\nAlready done : {len(already_done)} years")
print(f"Still needed : {len(to_download)} years")

  ⏳ 1998 needed
  ⏳ 1999 needed
  ⏳ 2000 needed
  ⏳ 2001 needed
  ⏳ 2002 needed
  ⏳ 2003 needed
  ⏳ 2004 needed
  ⏳ 2005 needed
  ⏳ 2006 needed
  ⏳ 2007 needed
  ⏳ 2008 needed
  ⏳ 2009 needed
  ⏳ 2010 needed
  ⏳ 2011 needed
  ⏳ 2012 needed
  ⏳ 2013 needed
  ⏳ 2014 needed
  ⏳ 2015 needed
  ⏳ 2016 needed
  ⏳ 2017 needed
  ⏳ 2018 needed
  ⏳ 2019 needed
  ⏳ 2020 needed

Already done : 0 years
Still needed : 23 years


In [8]:
import os
import gc
import xarray as xr

# set up credentials first
cds_config = """url: https://cds.climate.copernicus.eu/api
key: 3c560ea8-3044-4acb-80b0-2e8603a360b3
"""
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(cds_config)

# verify the file was written correctly
with open(os.path.expanduser('~/.cdsapirc'), 'r') as f:
    print("credentials file:")
    print(f.read())

# now import and start client
!pip install cdsapi -q
import cdsapi
c = cdsapi.Client()
print("client ready")

# download loop
years_all = list(range(1998, 2021))
hours     = [f'{h:02d}:00' for h in range(24)]
failed    = []

for year in years_all:
    
    day_file = f'/kaggle/working/era5_daily_{year}.nc'
    
    if os.path.exists(day_file):
        size_mb = os.path.getsize(day_file) / (1024*1024)
        print(f"{year} already done ({size_mb:.1f} MB) — skipping")
        continue
    
    raw_file = f'/kaggle/working/era5_hourly_{year}.nc'
    
    try:
        print(f"\n{year} downloading...")
        c.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type' : 'reanalysis',
                'variable'     : 'total_precipitation',
                'year'         : str(year),
                'month'        : ['06', '07', '08', '09'],
                'day'          : [str(d).zfill(2) for d in range(1, 32)],
                'time'         : hours,
                'format'       : 'netcdf',
                'area'         : [40, 60, 5, 100],
                'grid'         : [1.0, 1.0],
            },
            raw_file
        )
        
        ds_hr  = xr.open_dataset(raw_file)
        ds_day = ds_hr['tp'].resample(valid_time='1D').sum() * 1000
        
        print(f"{year} shape : {ds_day.shape}")
        print(f"{year} max   : {float(ds_day.max()):.2f} mm/day")
        print(f"{year} mean  : {float(ds_day.mean()):.2f} mm/day")
        
        ds_day.to_netcdf(day_file)
        size_mb = os.path.getsize(day_file) / (1024*1024)
        print(f"{year} saved ({size_mb:.1f} MB) checkpoint ✓")
        
        ds_hr.close()
        os.remove(raw_file)
        del ds_hr, ds_day
        gc.collect()
        
    except Exception as e:
        print(f"{year} failed: {e}")
        failed.append(year)
        if os.path.exists(raw_file):
            os.remove(raw_file)
        continue

print(f"\nfinished")
print(f"failed: {failed if failed else 'none'}")

credentials file:
url: https://cds.climate.copernicus.eu/api
key: 3c560ea8-3044-4acb-80b0-2e8603a360b3

client ready
1998 already done (0.7 MB) — skipping
1999 already done (0.7 MB) — skipping
2000 already done (0.7 MB) — skipping
2001 already done (0.7 MB) — skipping
2002 already done (0.7 MB) — skipping
2003 already done (0.7 MB) — skipping
2004 already done (0.7 MB) — skipping
2005 already done (0.7 MB) — skipping
2006 already done (0.7 MB) — skipping
2007 already done (0.7 MB) — skipping
2008 already done (0.7 MB) — skipping
2009 already done (0.7 MB) — skipping
2010 already done (0.7 MB) — skipping

2011 downloading...


2026-05-10 10:50:52,070 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-10 10:50:52,071 INFO Request ID is 995206a5-e27c-4077-9256-3f02c79e37d9
2026-05-10 10:50:52,856 INFO status has been updated to accepted
2026-05-10 10:57:13,635 INFO status has been updated to successful


eba2d8e1003d7819b124b32d039261f0.nc:   0%|          | 0.00/6.53M [00:00<?, ?B/s]

2011 shape : (122, 36, 41)
2011 max   : 379.03 mm/day
2011 mean  : 5.89 mm/day
2011 saved (0.7 MB) checkpoint ✓

2012 downloading...


2026-05-10 10:57:23,789 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-10 10:57:23,790 INFO Request ID is 6f23743a-a41c-4fd6-9bfe-e5010f630ba8
2026-05-10 10:57:23,938 INFO status has been updated to accepted
2026-05-10 10:57:46,195 INFO status has been updated to running
2026-05-10 11:01:44,794 INFO status has been updated to successful


...

3f8fd18ee33d12f3f12ec462e56d2127.nc:   0%|          | 0.00/5.98M [00:00<?, ?B/s]

2019 shape : (122, 36, 41)
2019 max   : 468.77 mm/day
2019 mean  : 5.84 mm/day
2019 saved (0.7 MB) checkpoint ✓

2020 downloading...


2026-05-10 11:39:29,954 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-10 11:39:29,955 INFO Request ID is 75c4ddaf-97cf-46d0-806f-014fa8c41826
2026-05-10 11:39:30,188 INFO status has been updated to accepted
2026-05-10 11:39:44,373 INFO status has been updated to running
2026-05-10 11:43:50,789 INFO status has been updated to successful


cfd51c2c0057495337e4aa6cce9ce053.nc:   0%|          | 0.00/6.44M [00:00<?, ?B/s]

2020 shape : (122, 36, 41)
2020 max   : 307.81 mm/day
2020 mean  : 5.96 mm/day
2020 saved (0.7 MB) checkpoint ✓

finished
failed: none


In [9]:
import xarray as xr
import numpy as np
import os

# check all yearly files exist
print("=== FILE CHECK ===")
total_size = 0
missing = []

for year in range(1998, 2021):
    f = f'/kaggle/working/era5_daily_{year}.nc'
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024*1024)
        total_size += size_mb
        print(f"{year} — {size_mb:.1f} MB")
    else:
        missing.append(year)
        print(f"{year} — MISSING")

print(f"\ntotal size : {total_size:.1f} MB")
print(f"missing    : {missing if missing else 'none'}")

# load and check each year
print("\n=== PER YEAR STATS ===")
print(f"{'year':<6} {'days':<6} {'max':>10} {'mean':>10}")
print("-" * 35)

for year in range(1998, 2021):
    f = f'/kaggle/working/era5_daily_{year}.nc'
    if os.path.exists(f):
        ds = xr.open_dataarray(f)
        n_days = len(ds.valid_time)
        mx     = float(ds.max())
        mn     = float(ds.mean())
        print(f"{year:<6} {n_days:<6} {mx:>10.2f} {mn:>10.2f}")
        ds.close()

# load one year and do a physical check
print("\n=== PHYSICAL CHECK — 1998 ===")
ds = xr.open_dataarray('/kaggle/working/era5_daily_1998.nc')

# check seasonal cycle — july/august should be wetter than june/september
import pandas as pd
times = pd.to_datetime(ds.valid_time.values)
data  = ds.values  # (days, lat, lon)

june = data[times.month == 6].mean()
july = data[times.month == 7].mean()
aug  = data[times.month == 8].mean()
sept = data[times.month == 9].mean()

print(f"June mean  : {june:.2f} mm/day")
print(f"July mean  : {july:.2f} mm/day")
print(f"August mean: {aug:.2f} mm/day")
print(f"Sept mean  : {sept:.2f} mm/day")
print(f"July/August should be higher than June/September")

ds.close()

# check a known active monsoon year — 2007 should have high values
print("\n=== ACTIVE YEAR CHECK ===")
print("2007 and 2010 should have higher max/mean than 1999 and 2002")

for year in [1999, 2002, 2007, 2010]:
    f = f'/kaggle/working/era5_daily_{year}.nc'
    if os.path.exists(f):
        ds  = xr.open_dataarray(f)
        mx  = float(ds.max())
        mn  = float(ds.mean())
        print(f"{year} — max {mx:.1f}  mean {mn:.2f}")
        ds.close()

=== FILE CHECK ===
1998 — 0.7 MB
1999 — 0.7 MB
2000 — 0.7 MB
2001 — 0.7 MB
2002 — 0.7 MB
2003 — 0.7 MB
2004 — 0.7 MB
2005 — 0.7 MB
2006 — 0.7 MB
2007 — 0.7 MB
2008 — 0.7 MB
2009 — 0.7 MB
2010 — 0.7 MB
2011 — 0.7 MB
2012 — 0.7 MB
2013 — 0.7 MB
2014 — 0.7 MB
2015 — 0.7 MB
2016 — 0.7 MB
2017 — 0.7 MB
2018 — 0.7 MB
2019 — 0.7 MB
2020 — 0.7 MB

total size : 16.3 MB
missing    : none

=== PER YEAR STATS ===
year   days          max       mean
-----------------------------------
1998   122        343.51       5.73
1999   122        290.40       4.90
2000   122        362.95       5.26
2001   122        335.90       5.09
2002   122        362.28       4.72
2003   122        344.92       5.70
2004   122        321.25       5.30
2005   122        311.60       5.44
2006   122        385.08       5.51
2007   122        492.96       6.32
2008   122        377.90       5.60
2009   122        374.20       5.15
2010   122        334.91       6.09
2011   122        379.03       5.89
2012   122        3

In [10]:
import xarray as xr
import os

datasets = []

for year in range(1998, 2021):
    f = f'/kaggle/working/era5_daily_{year}.nc'
    ds = xr.open_dataarray(f)
    datasets.append(ds)
    print(f"loaded {year}")

ds_all   = xr.concat(datasets, dim='valid_time')
ds_all   = ds_all.rename({'valid_time': 'time', 
                           'latitude' : 'lat', 
                           'longitude': 'lon'})
ds_final = xr.Dataset({'precipitation': ds_all})

print(f"\nshape      : {ds_final['precipitation'].shape}")
print(f"time range : {str(ds_final.time.values[0])[:10]} to {str(ds_final.time.values[-1])[:10]}")
print(f"max        : {float(ds_final.precipitation.max()):.2f} mm/day")
print(f"mean       : {float(ds_final.precipitation.mean()):.2f} mm/day")

ds_final.to_netcdf('/kaggle/working/era5_india_daily_correct.nc')
print("\nsaved → era5_india_daily_correct.nc")

loaded 1998
loaded 1999
loaded 2000
loaded 2001
loaded 2002
loaded 2003
loaded 2004
loaded 2005
loaded 2006
loaded 2007
loaded 2008
loaded 2009
loaded 2010
loaded 2011
loaded 2012
loaded 2013
loaded 2014
loaded 2015
loaded 2016
loaded 2017
loaded 2018
loaded 2019
loaded 2020

shape      : (2806, 36, 41)
time range : 1998-06-01 to 2020-09-30
max        : 492.96 mm/day
mean       : 5.45 mm/day

saved → era5_india_daily_correct.nc


In [11]:
import xarray as xr
import numpy as np
import pandas as pd

ds = xr.open_dataset('/kaggle/working/era5_india_daily_correct.nc')

print("=== BASIC STRUCTURE ===")
print(ds)
print()

print("=== KEY NUMBERS ===")
print(f"shape      : {ds['precipitation'].shape}")
print(f"time range : {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
print(f"total days : {len(ds.time)}")
print(f"lat range  : {float(ds.lat.min())} to {float(ds.lat.max())}")
print(f"lon range  : {float(ds.lon.min())} to {float(ds.lon.max())}")
print()

print("=== PRECIPITATION VALUES ===")
print(f"max  : {float(ds.precipitation.max()):.2f} mm/day")
print(f"mean : {float(ds.precipitation.mean()):.2f} mm/day")
print(f"min  : {float(ds.precipitation.min()):.2f} mm/day")
print()

# check all 23 years are present
times = pd.to_datetime(ds.time.values)
years_present = sorted(times.year.unique())
print(f"=== YEARS PRESENT ===")
print(years_present)
print(f"count : {len(years_present)}")
print()

# check day counts per year
print("=== DAYS PER YEAR ===")
for yr in years_present:
    n = (times.year == yr).sum()
    print(f"{yr} : {n} days")

ds.close()

=== BASIC STRUCTURE ===
<xarray.Dataset> Size: 17MB
Dimensions:        (time: 2806, lat: 36, lon: 41)
Coordinates:
  * time           (time) datetime64[ns] 22kB 1998-06-01 ... 2020-09-30
  * lat            (lat) float64 288B 40.0 39.0 38.0 37.0 ... 8.0 7.0 6.0 5.0
  * lon            (lon) float64 328B 60.0 61.0 62.0 63.0 ... 98.0 99.0 100.0
    number         int64 8B ...
Data variables:
    precipitation  (time, lat, lon) float32 17MB ...

=== KEY NUMBERS ===
shape      : (2806, 36, 41)
time range : 1998-06-01 to 2020-09-30
total days : 2806
lat range  : 5.0 to 40.0
lon range  : 60.0 to 100.0

=== PRECIPITATION VALUES ===
max  : 492.96 mm/day
mean : 5.45 mm/day
min  : 0.00 mm/day

=== YEARS PRESENT ===
[1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
count : 23

=== DAYS PER YEAR ===
1998 : 122 days
1999 : 122 days
2000 : 122 days
2001 : 122 days
2002 : 122 days
2003 : 122 days
2004 : 122 days
20

In [13]:
import os

# search for the file
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/divyanshyecho/era5-india-daily-precip/era5_india_daily_correct.nc


In [1]:
import xarray as xr

ds = xr.open_dataset('/kaggle/input/datasets/divyanshyecho/era5-india-daily-precip/era5_india_daily_correct.nc')

print(f"shape      : {ds['precipitation'].shape}")
print(f"time range : {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
print(f"max        : {float(ds.precipitation.max()):.2f} mm/day")
print(f"mean       : {float(ds.precipitation.mean()):.2f} mm/day")

ds.close()
print("\ndataset loaded successfully")

shape      : (2806, 36, 41)
time range : 1998-06-01 to 2020-09-30
max        : 492.96 mm/day
mean       : 5.45 mm/day

dataset loaded successfully


The dataset has been loaded, 1998-2020 , june 1st to sep 30th , daily precipitation data in mm/day. Now making the thresholding model for ERE.

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
from scipy import ndimage

ds = xr.open_dataset('/kaggle/input/datasets/divyanshyecho/era5-india-daily-precip/era5_india_daily_correct.nc')

data_precip = np.array(ds['precipitation'][:])
lat         = ds['lat'].values
lon         = ds['lon'].values
time        = ds['time'].values

print(f"shape : {data_precip.shape}")
print(f"time  : {str(time[0])[:10]} to {str(time[-1])[:10]}")
print(f"max   : {np.nanmax(data_precip):.2f} mm/day")
print(f"mean  : {np.nanmean(data_precip):.2f} mm/day")

shape : (2806, 36, 41)
time  : 1998-06-01 to 2020-09-30
max   : 492.96 mm/day
mean  : 5.47 mm/day


In [3]:
data_precip1 = np.copy(data_precip)
data_precip1[data_precip1 <= 0.0] = np.nan

rainy = np.count_nonzero(~np.isnan(data_precip1))
dry   = np.count_nonzero(np.isnan(data_precip1))

print(f"rainy grid-days : {rainy}")
print(f"dry grid-days   : {dry}")
print(f"rainy fraction  : {rainy/(rainy+dry)*100:.1f}%")

rainy grid-days : 3523080
dry grid-days   : 618576
rainy fraction  : 85.1%


In [5]:
per_grid = np.full((len(lat), len(lon)), np.nan, dtype='float32')

for x in range(len(lat)):
    for y in range(len(lon)):
        col = data_precip1[:, x, y]
        col = col[~np.isnan(col)]
        if len(col) > 0:
            per_grid[x, y] = np.percentile(col, 99)

print(f"threshold shape : {per_grid.shape}")
print(f"min threshold   : {np.nanmin(per_grid):.2f} mm/day")
print(f"max threshold   : {np.nanmax(per_grid):.2f} mm/day")
print(f"mean threshold  : {np.nanmean(per_grid):.2f} mm/day")

threshold shape : (36, 41)
min threshold   : 0.31 mm/day
max threshold   : 162.87 mm/day
mean threshold  : 35.36 mm/day


In [6]:
prep_thrsh = np.empty_like(data_precip)
for t in range(len(time)):
    prep_thrsh[t, :, :] = per_grid[:, :]

data_precip_safe = np.copy(data_precip)
data_precip_safe[np.isnan(data_precip_safe)] = 0.0
prep_thrsh[np.isnan(prep_thrsh)]             = 999.0

c = (data_precip_safe > prep_thrsh)

print(f"exceedance mask shape   : {c.shape}")
print(f"total extreme grid-days : {c.sum()}")
print(f"fraction exceeding      : {c.mean()*100:.2f}%")

exceedance mask shape   : (2806, 36, 41)
total extreme grid-days : 35962
fraction exceeding      : 0.87%


In [7]:
Loc_ERE       = np.zeros((len(time), len(lat), len(lon)), dtype='float32')
nt_per_day    = []
n_ere_per_day = []

for k in range(len(time)):
    img = c[k, :, :].astype(float)

    labeled, nr_objects = ndimage.label(img)

    nt_per_day.append(int(c[k, :, :].sum()))
    n_ere_per_day.append(nr_objects)

    if nr_objects >= 1:
        sfc_areas = np.bincount(labeled.flat)[1:]
        for i in range(nr_objects):
            sz  = sfc_areas[i]
            ind = np.where(labeled == i + 1)
            Loc_ERE[k, ind[0], ind[1]] = sz

    day_slice = Loc_ERE[k, :, :]
    day_slice[day_slice == 0.0] = np.nan
    Loc_ERE[k, :, :] = day_slice

    if k % 500 == 0:
        print(f"day {k}/{len(time)}")

print(f"\ndetection complete")
print(f"mean EREs per day : {np.mean(n_ere_per_day):.2f}")
print(f"max EREs one day  : {np.max(n_ere_per_day)}")
print(f"mean NT per day   : {np.mean(nt_per_day):.2f}")

day 0/2806
day 500/2806
day 1000/2806
day 1500/2806
day 2000/2806
day 2500/2806

detection complete
mean EREs per day : 5.48
max EREs one day  : 18
mean NT per day   : 12.82


In [8]:
small  = (Loc_ERE == 1)
medium = (Loc_ERE >= 2) & (Loc_ERE <= 5)
large  = (Loc_ERE >= 6)

print("=== SIZE CLASSIFICATION ===")
print(f"small  (size=1)   : {small.sum()}")
print(f"medium (size 2-5) : {medium.sum()}")
print(f"large  (size>=6)  : {large.sum()}")

df = pd.DataFrame({
    'date' : pd.to_datetime(time),
    'NT'   : nt_per_day,
    'NE'   : n_ere_per_day,
})
df['year']  = df['date'].dt.year
df['S_bar'] = df['NT'] / df['NE'].replace(0, np.nan)

annual = df.groupby('year')[['NT', 'NE', 'S_bar']].mean().reset_index()
print("\n=== ANNUAL STATS ===")
print(annual.to_string(index=False))

=== SIZE CLASSIFICATION ===
small  (size=1)   : 10004
medium (size 2-5) : 11315
large  (size>=6)  : 14643

=== ANNUAL STATS ===
 year        NT       NE    S_bar
 1998 12.713115 6.131148 2.203642
 1999  9.754098 5.172131 1.914378
 2000 11.901639 4.934426 2.529542
 2001  9.213115 4.704918 1.954124
 2002  9.852459 4.852459 2.181937
 2003 12.393443 5.172131 2.403171
 2004 10.434426 4.352459 2.172384
 2005 12.704918 5.475410 2.429827
 2006 14.704918 5.565574 2.757377
 2007 20.524590 6.467213 3.158982
 2008 11.680328 5.327869 2.430492
 2009 10.827869 5.032787 2.290899
 2010 18.450820 6.581967 2.772953
 2011 13.213115 5.483607 2.591892
 2012  9.237705 4.754098 2.095197
 2013 12.221311 5.844262 2.057736
 2014 12.262295 5.114754 2.926502
 2015 15.803279 5.598361 3.323014
 2016 11.106557 5.393443 2.175019
 2017 12.532787 5.975410 2.193629
 2018 10.057377 5.344262 1.855149
 2019 16.803279 6.622951 2.654789
 2020 16.377049 6.032787 2.867961


In [9]:
# These are well documented extreme rainfall events over India
# ERA5 should flag them as extreme even if values are smoothed

known_events = {
    '2005-07-26': (19, 73, 'Mumbai floods'),
    '2013-06-16': (30, 79, 'Uttarakhand floods'),
    '2007-07-15': (25, 85, 'Bihar floods'),
    '2010-08-06': (34, 74, 'Leh cloudbursts'),
}

print("=== KNOWN EXTREME EVENT CHECK ===\n")

times_pd = pd.to_datetime(time)

for date_str, (target_lat, target_lon, name) in known_events.items():
    
    date = pd.Timestamp(date_str)
    
    # check if date is in our dataset
    idx = np.where(times_pd == date)[0]
    if len(idx) == 0:
        print(f"{name} ({date_str}) — not in dataset (outside monsoon season or year range)")
        continue
    
    idx = idx[0]
    
    # find nearest grid cell
    lat_idx = np.argmin(np.abs(lat - target_lat))
    lon_idx = np.argmin(np.abs(lon - target_lon))
    
    actual_rain = data_precip[idx, lat_idx, lon_idx]
    threshold   = per_grid[lat_idx, lon_idx]
    exceeded    = actual_rain > threshold
    ere_size    = Loc_ERE[idx, lat_idx, lon_idx]
    
    print(f"{name} ({date_str})")
    print(f"  nearest grid    : {lat[lat_idx]}N {lon[lon_idx]}E")
    print(f"  rainfall        : {actual_rain:.2f} mm/day")
    print(f"  threshold       : {threshold:.2f} mm/day")
    print(f"  exceeded        : {exceeded}")
    print(f"  ERE size        : {ere_size if not np.isnan(ere_size) else 'not part of ERE'}")
    print(f"  NT on this day  : {nt_per_day[idx]}")
    print(f"  NE on this day  : {n_ere_per_day[idx]}")
    print()

=== KNOWN EXTREME EVENT CHECK ===

Mumbai floods (2005-07-26)
  nearest grid    : 19.0N 73.0E
  rainfall        : 71.65 mm/day
  threshold       : 107.35 mm/day
  exceeded        : False
  ERE size        : not part of ERE
  NT on this day  : 9
  NE on this day  : 8

Uttarakhand floods (2013-06-16)
  nearest grid    : 30.0N 79.0E
  rainfall        : 151.13 mm/day
  threshold       : 87.10 mm/day
  exceeded        : True
  ERE size        : 19.0
  NT on this day  : 53
  NE on this day  : 9

Bihar floods (2007-07-15)
  nearest grid    : 25.0N 85.0E
  rainfall        : 20.40 mm/day
  threshold       : 51.48 mm/day
  exceeded        : False
  ERE size        : not part of ERE
  NT on this day  : 13
  NE on this day  : 4

Leh cloudbursts (2010-08-06)
  nearest grid    : 34.0N 74.0E
  rainfall        : 30.30 mm/day
  threshold       : 46.31 mm/day
  exceeded        : False
  ERE size        : not part of ERE
  NT on this day  : 16
  NE on this day  : 12



In [10]:
# july and august should have more extreme events than june and september
# this is a basic physical consistency check

print("=== SEASONAL CYCLE CHECK ===\n")

times_pd  = pd.to_datetime(time)
month_map = {6: 'June', 7: 'July', 8: 'August', 9: 'September'}

df_daily = pd.DataFrame({
    'date'  : times_pd,
    'month' : times_pd.month,
    'NT'    : nt_per_day,
    'NE'    : n_ere_per_day,
})
df_daily['S_bar'] = df_daily['NT'] / df_daily['NE'].replace(0, np.nan)

monthly = df_daily.groupby('month')[['NT', 'NE', 'S_bar']].mean()

print(f"{'month':<12} {'NT':>8} {'NE':>8} {'S_bar':>8}")
print("-" * 40)
for m in [6, 7, 8, 9]:
    print(f"{month_map[m]:<12} {monthly.loc[m,'NT']:>8.2f} {monthly.loc[m,'NE']:>8.2f} {monthly.loc[m,'S_bar']:>8.2f}")

print()
print("expected: July and August should have highest NT and NE")

# check if july/august peak
nt_values  = [monthly.loc[m, 'NT'] for m in [6,7,8,9]]
peak_month = [6,7,8,9][np.argmax(nt_values)]
print(f"actual peak month: {month_map[peak_month]}")
if peak_month in [7, 8]:
    print("seasonal cycle: PASS")
else:
    print("seasonal cycle: FAIL — check threshold calculation")

=== SEASONAL CYCLE CHECK ===

month              NT       NE    S_bar
----------------------------------------
June            15.93     5.21     3.27
July            13.12     6.70     1.97
August          11.51     5.74     2.02
September       10.74     4.21     2.52

expected: July and August should have highest NT and NE
actual peak month: June
seasonal cycle: FAIL — check threshold calculation


In [11]:
# well documented active and weak monsoon years
# active years should have higher NT, NE and S_bar

print("=== ACTIVE VS WEAK MONSOON YEAR CHECK ===\n")

# documented weak monsoon years (drought/El Nino)
weak_years   = [2002, 2004, 2009, 2014, 2015]

# documented active monsoon years
active_years = [2007, 2010, 2011, 2019, 2020]

annual_stats = df.groupby('year')[['NT', 'NE', 'S_bar']].mean()

print("weak monsoon years:")
print(f"{'year':<6} {'NT':>8} {'NE':>8} {'S_bar':>8}")
print("-" * 35)
for yr in weak_years:
    if yr in annual_stats.index:
        row = annual_stats.loc[yr]
        print(f"{yr:<6} {row['NT']:>8.2f} {row['NE']:>8.2f} {row['S_bar']:>8.2f}")

print()
print("active monsoon years:")
print(f"{'year':<6} {'NT':>8} {'NE':>8} {'S_bar':>8}")
print("-" * 35)
for yr in active_years:
    if yr in annual_stats.index:
        row = annual_stats.loc[yr]
        print(f"{yr:<6} {row['NT']:>8.2f} {row['NE']:>8.2f} {row['S_bar']:>8.2f}")

print()
weak_mean_NT   = annual_stats.loc[annual_stats.index.isin(weak_years), 'NT'].mean()
active_mean_NT = annual_stats.loc[annual_stats.index.isin(active_years), 'NT'].mean()
print(f"mean NT weak years   : {weak_mean_NT:.2f}")
print(f"mean NT active years : {active_mean_NT:.2f}")

if active_mean_NT > weak_mean_NT:
    print("active vs weak check : PASS")
else:
    print("active vs weak check : FAIL")

=== ACTIVE VS WEAK MONSOON YEAR CHECK ===

weak monsoon years:
year         NT       NE    S_bar
-----------------------------------
2002       9.85     4.85     2.18
2004      10.43     4.35     2.17
2009      10.83     5.03     2.29
2014      12.26     5.11     2.93
2015      15.80     5.60     3.32

active monsoon years:
year         NT       NE    S_bar
-----------------------------------
2007      20.52     6.47     3.16
2010      18.45     6.58     2.77
2011      13.21     5.48     2.59
2019      16.80     6.62     2.65
2020      16.38     6.03     2.87

mean NT weak years   : 11.84
mean NT active years : 17.07
active vs weak check : PASS


In [13]:
print("=== SPATIAL PATTERN CHECK ===\n")

ere_freq = (~np.isnan(Loc_ERE)).sum(axis=0)

regions = {
    'Western Ghats'   : {'lat_min': 8,  'lat_max': 20, 'lon_min': 72, 'lon_max': 76},
    'Central India'   : {'lat_min': 15, 'lat_max': 25, 'lon_min': 75, 'lon_max': 85},
    'Northeast India' : {'lat_min': 22, 'lat_max': 30, 'lon_min': 88, 'lon_max': 97},
    'Northwest India' : {'lat_min': 25, 'lat_max': 35, 'lon_min': 68, 'lon_max': 76},
    'Pakistan/Desert' : {'lat_min': 25, 'lat_max': 35, 'lon_min': 60, 'lon_max': 68},
}

for region_name, bounds in regions.items():
    lat_idx = np.where((lat >= bounds['lat_min']) & (lat <= bounds['lat_max']))[0]
    lon_idx = np.where((lon >= bounds['lon_min']) & (lon <= bounds['lon_max']))[0]
    
    region_freq = ere_freq[np.ix_(lat_idx, lon_idx)].mean()
    print(f"{region_name:<20} : {region_freq:.1f} days with ERE")

print()
print("expected: Western Ghats and Central India highest")
print("expected: Pakistan/Desert lowest")

=== SPATIAL PATTERN CHECK ===

Western Ghats        : 28.6 days with ERE
Central India        : 27.4 days with ERE
Northeast India      : 28.6 days with ERE
Northwest India      : 20.7 days with ERE
Pakistan/Desert      : 7.4 days with ERE

expected: Western Ghats and Central India highest
expected: Pakistan/Desert lowest


In [14]:
# the 99th percentile threshold should be higher in wet regions
# and lower in dry regions
# also check fraction exceeding is close to 1%

print("=== THRESHOLD SANITY CHECK ===\n")

# fraction of rainy days exceeding threshold per grid
# should be close to 1% everywhere by definition
exceedance_rate = c.mean(axis=0) * 100  # percentage

print(f"mean exceedance rate across all grids : {exceedance_rate.mean():.2f}%")
print(f"max exceedance rate at any grid       : {exceedance_rate.max():.2f}%")
print(f"min exceedance rate at any grid       : {exceedance_rate.min():.2f}%")
print(f"expected: all values close to 1%")
print()

# threshold should correlate with mean rainfall
mean_rain = np.nanmean(data_precip, axis=0)
corr = np.corrcoef(mean_rain.flatten(), per_grid.flatten())[0,1]
print(f"correlation between mean rain and threshold : {corr:.3f}")
print(f"expected: high positive correlation (>0.7)")
if corr > 0.7:
    print("threshold spatial pattern : PASS")
else:
    print("threshold spatial pattern : CHECK — lower than expected")
print()

# wet vs dry region thresholds
wet_threshold = per_grid[
    np.ix_(np.where((lat >= 10) & (lat <= 20))[0],
           np.where((lon >= 72) & (lon <= 76))[0])
].mean()

dry_threshold = per_grid[
    np.ix_(np.where((lat >= 25) & (lat <= 35))[0],
           np.where((lon >= 60) & (lon <= 68))[0])
].mean()

print(f"mean threshold Western Ghats : {wet_threshold:.2f} mm/day")
print(f"mean threshold Pakistan/dry  : {dry_threshold:.2f} mm/day")
print(f"wet region should have higher threshold than dry region")
if wet_threshold > dry_threshold:
    print("wet vs dry threshold : PASS")
else:
    print("wet vs dry threshold : FAIL")

=== THRESHOLD SANITY CHECK ===

mean exceedance rate across all grids : 0.87%
max exceedance rate at any grid       : 1.03%
min exceedance rate at any grid       : 0.04%
expected: all values close to 1%

correlation between mean rain and threshold : 0.839
expected: high positive correlation (>0.7)
threshold spatial pattern : PASS

mean threshold Western Ghats : 57.78 mm/day
mean threshold Pakistan/dry  : 13.62 mm/day
wet region should have higher threshold than dry region
wet vs dry threshold : PASS


The 99 percentile must be month based and not all the months should be used together. More Nt for the month of june.